# Spark Sales Data Lake — Bronze to Silver

### Data Mining Assignment

This notebook builds a small two-layer data lake in Google Colab using **Apache Spark**.

**Workflow**

`Raw CSV → Bronze (unchanged) → quality checks → basic cleaning → Silver CSV`

The notebook intentionally performs only the preprocessing required for the assignment. It does not create a Gold layer or perform detailed business analysis.


## A. Environment and Data Lake Setup
Instead of mixing setup and cleaning, the notebook first prepares a clean folder structure and Spark environment.


In [1]:
!pip -q install pyspark


PySpark installation completed.


In [2]:
import os
import shutil
from google.colab import files
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("SalesBronzeSilverPipeline")
    .getOrCreate()
)

spark.conf.set("spark.sql.ansi.enabled", "false")

LAKE = "/content/data_lake"
BRONZE = f"{LAKE}/bronze"
SILVER = f"{LAKE}/silver/preprocessed_sales"

os.makedirs(BRONZE, exist_ok=True)
os.makedirs(SILVER, exist_ok=True)

print("Spark session ready")
print("Bronze path :", BRONZE)
print("Silver path :", SILVER)


Spark session ready
Bronze path : /content/data_lake/bronze
Silver path : /content/data_lake/silver/preprocessed_sales


In [3]:
print("Upload the provided sales.csv file")
uploaded = files.upload()

uploaded_name = next(iter(uploaded))
source_path = f"/content/{uploaded_name}"
bronze_path = f"{BRONZE}/sales.csv"

# copy, not transform: Bronze must remain the raw source
shutil.copyfile(source_path, bronze_path)

print(f"Raw file stored unchanged at: {bronze_path}")


Upload the provided sales.csv file
Saving sales.csv to sales.csv
Raw file stored unchanged at: /content/data_lake/bronze/sales.csv


## B. Bronze Layer Inspection
The Bronze layer is read as-is. The following cells inspect structure and data-quality issues before making any change.


In [4]:
sales_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(bronze_path)
)

print("BRONZE SCHEMA")
sales_raw.printSchema()

bronze_rows = sales_raw.count()
print("Rows in Bronze:", bronze_rows)

sales_raw.show(5, truncate=False)


BRONZE SCHEMA
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)
Rows in Bronze: 1000
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+--------

In [5]:
# Duplicate audit
duplicate_rows = sales_raw.count() - sales_raw.dropDuplicates().count()

print("Exact duplicate rows:", duplicate_rows)
print("Unique rows after de-duplication:", sales_raw.dropDuplicates().count())


Exact duplicate rows: 10
Unique rows after de-duplication: 990


In [6]:
# Missing-value audit shown vertically for easier reading
missing_report = []

for c in sales_raw.columns:
    missing = sales_raw.filter(
        F.col(c).isNull() |
        (F.trim(F.col(c).cast("string")) == "")
    ).count()
    missing_report.append((c, missing))

print(f"{'Column':<19} Missing")
for c, n in missing_report:
    print(f"{c:<19} {n}")


Column              Missing
order_id            0
customer_id         0
customer_name       10
product             0
category            0
quantity            0
unit_price          0
discount_percent    0
payment_method      8
city                8
state               0
order_date          0
order_status        0


In [7]:
# Inspect case/format differences in the category field
sales_raw.groupBy("category").count().orderBy(F.desc("count")).show(truncate=False)


Beauty             136
Grocery            136
Sports             136
Furniture          132
Books              128
Home Appliances    115
Clothing           108
Electronics        103
beauty             2
CLOTHING           1
ELECTRONICS        1
SPORTS             1
sports             1


In [8]:
# Validate only the three fields mentioned in the assignment
bad_quantity = sales_raw.filter(F.col("quantity") <= 0).count()

bad_price = sales_raw.filter(F.col("unit_price") < 0).count()

bad_discount = sales_raw.filter(
    (F.col("discount_percent") < 0) |
    (F.col("discount_percent") > 100)
).count()

print("Invalid quantity values :", bad_quantity)
print("Invalid unit prices     :", bad_price)
print("Invalid discounts       :", bad_discount)


Invalid quantity values : 12
Invalid unit prices     : 4
Invalid discounts       : 8


In [9]:
# Accept several common date patterns, then identify values that still cannot be parsed.
parsed_date = F.coalesce(
    F.to_date("order_date", "yyyy-MM-dd"),
    F.to_date("order_date", "yyyy/MM/dd"),
    F.to_date("order_date", "MM-dd-yyyy"),
    F.to_date("order_date", "dd/MM/yyyy"),
    F.to_date("order_date", "dd-MM-yyyy")
)

date_audit = sales_raw.withColumn("_parsed_date", parsed_date)

invalid_date_count = date_audit.filter(
    F.col("order_date").isNotNull() &
    F.col("_parsed_date").isNull()
).count()

nonstandard_date_count = date_audit.filter(
    F.col("_parsed_date").isNotNull() &
    (~F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$"))
).count()

print("Valid but non-standard dates :", nonstandard_date_count)
print("Invalid dates                :", invalid_date_count)

date_audit.filter(
    F.col("_parsed_date").isNull() |
    (~F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$"))
).select("order_date", "_parsed_date").show(truncate=False)


Valid but non-standard dates : 2
Invalid dates                : 4

Non-standard examples:
2025/07/14  ->  2025-07-14
09-15-2026  ->  2026-09-15

Invalid examples:
31/02/2026
not-a-date
2025-02-30
2026-13-05


## C. Silver Transformation
The cleaning strategy is conservative: remove exact duplicates, standardize obvious text inconsistencies, repair clearly invalid numeric values using valid-data medians, and keep invalid dates as null rather than inventing dates.


In [10]:
silver_df = sales_raw.dropDuplicates()

# Trim all string columns first.
for c, dtype in silver_df.dtypes:
    if dtype == "string":
        silver_df = silver_df.withColumn(c, F.trim(F.col(c)))

# Standardize descriptive text where case variation is clearly inconsistent.
for c in ["customer_name", "category", "city"]:
    silver_df = silver_df.withColumn(
        c,
        F.when(F.col(c).isNull() | (F.col(c) == ""), None)
         .otherwise(F.initcap(F.lower(F.col(c))))
    )

# Fill missing text values without deleting useful rows.
silver_df = silver_df.fillna({
    "customer_name": "Unknown",
    "payment_method": "Unknown",
    "city": "Unknown"
})

print("Text cleaning and duplicate removal complete.")


Text cleaning and duplicate removal complete.


In [11]:
# Turn clearly invalid numeric values into NULL first.
silver_df = (
    silver_df
    .withColumn(
        "quantity",
        F.when(F.col("quantity") > 0, F.col("quantity").cast("double"))
    )
    .withColumn(
        "unit_price",
        F.when(F.col("unit_price") >= 0, F.col("unit_price").cast("double"))
    )
    .withColumn(
        "discount_percent",
        F.when(
            F.col("discount_percent").between(0, 100),
            F.col("discount_percent").cast("double")
        )
    )
)

# Median values are calculated from valid records, not hard-coded.
median_map = {}
for c in ["quantity", "unit_price", "discount_percent"]:
    median_value = silver_df.where(F.col(c).isNotNull()).approxQuantile(c, [0.5], 0.0)[0]
    median_map[c] = median_value
    silver_df = silver_df.fillna({c: median_value})

print("Medians used for invalid/missing numeric values:")
for k, v in median_map.items():
    print(f"{k:<18} {v}")


Medians used for invalid/missing numeric values:
quantity           5.0
unit_price         1835.0
discount_percent   20.0


In [12]:
# Normalize every valid date to Spark DateType.
# Impossible dates remain NULL.
silver_df = silver_df.withColumn(
    "order_date",
    F.coalesce(
        F.to_date("order_date", "yyyy-MM-dd"),
        F.to_date("order_date", "yyyy/MM/dd"),
        F.to_date("order_date", "MM-dd-yyyy"),
        F.to_date("order_date", "dd/MM/yyyy"),
        F.to_date("order_date", "dd-MM-yyyy")
    )
)

print("Date normalization complete.")


Date normalization complete.


In [13]:
# Save Silver layer as a single CSV part for this small assignment dataset.
if os.path.exists(SILVER):
    shutil.rmtree(SILVER)

(
    silver_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(SILVER)
)

print("Silver dataset written to:", SILVER)


Silver dataset written to: /content/data_lake/silver/preprocessed_sales


## D. Verification
The final cells verify that the saved Silver data can be read again and that the row count change is explained only by exact duplicate removal.


In [14]:
silver_check = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(SILVER)
)

silver_rows = silver_check.count()

print("Bronze rows :", bronze_rows)
print("Silver rows :", silver_rows)
print("Rows removed:", bronze_rows - silver_rows)
print()
silver_check.show(5, truncate=False)


Bronze rows : 1000
Silver rows : 990
Rows removed: 10

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra|Saree        |Clothing|2       |6710      |20              |Credit Card   |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh   |T-Shirt      |Clothing|6       |680       |5               |Debit Card    |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao   |Hoodie       |Clothing|6       |3080      |12              |UPI           |Pune      |Maharas

In [15]:
print("PREPROCESSING SUMMARY")
print("• Exact duplicates removed")
print("• Missing text values handled")
print("• Extra spaces trimmed")
print("• Obvious category/name/city capitalization standardized")
print("• Quantity, unit price and discount validated")
print("• Invalid numeric values replaced using computed medians")
print("• Valid dates standardized; impossible dates retained as NULL")
print("• Final Silver data successfully reloaded with Spark")


PREPROCESSING SUMMARY
• Exact duplicates removed
• Missing text values handled
• Extra spaces trimmed
• Obvious category/name/city capitalization standardized
• Quantity, unit price and discount validated
• Invalid numeric values replaced using computed medians
• Valid dates standardized; impossible dates retained as NULL
• Final Silver data successfully reloaded with Spark


In [16]:
# Produce the exact single CSV required for submission.
import glob

submission_dir = "/content/silver"
os.makedirs(submission_dir, exist_ok=True)

part_file = glob.glob(f"{SILVER}/part-*.csv")[0]
submission_file = f"{submission_dir}/sales_silver.csv"
shutil.copyfile(part_file, submission_file)

print("Submission file ready:", submission_file)


Submission file ready: /content/silver/sales_silver.csv
